# Análise Exploratória dos Dados — CreditGuard AI v3
## ProScore Analytics — Predição de Inadimplência

**Disciplina:** Introdução à Ciência de Dados  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Arquivo analisado:** `Dados/clean_data.csv` — saída de `DataPipeline/data_sanitization.py` (v3)

---

### Objetivo

Este notebook apresenta a análise exploratória dos dados após a etapa de sanitização do projeto **CreditGuard AI**, sistema de predição de inadimplência em instituições financeiras. O objetivo é compreender a estrutura, qualidade e distribuições dos dados antes da modelagem preditiva, além de validar as decisões de feature engineering e conectar os achados ao modelo LightGBM v3 em produção.

O pipeline de dados combina duas fontes do Home Credit:
- `application_train.csv` — dados demográficos e financeiros da solicitação de crédito
- `bureau.csv` — histórico de crédito externo (agregado por cliente)

A variável alvo `TARGET = 1` indica que o cliente ficou inadimplente no empréstimo.

---

### Roteiro da Análise (14 seções)

1. Carregamento dos dados
2. Dimensionalidade e estrutura
3. Tipos de variáveis
4. Análise de valores ausentes
5. Estatísticas descritivas
6. Features derivadas — Application Train (feature engineering)
7. Distribuição da variável TARGET
8. Análise univariada das principais variáveis
9. Matriz de correlação
10. Insights — Análise bivariada
11. Feature Importance — LightGBM v3
12. Cobertura do formulário Streamlit
13. Features ausentes e estratégia de inferência
14. Conclusões

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 50)

## 1. Carregamento dos Dados

O arquivo `clean_data.csv` é gerado por `DataPipeline/data_sanitization.py`, que executa o seguinte pipeline:

1. Carrega `application_train.csv` (307.511 registros × ~120 colunas originais)
2. Aplica `create_application_features()` — cria 8 features derivadas (razões financeiras e temporais)
3. Carrega `bureau.csv` e agrega por `SK_ID_CURR` — gera features de histórico de crédito
4. Faz merge entre application e bureau aggregations
5. Salva `clean_data.csv` com **185 features brutas + TARGET** (186 colunas no total)

**Importante:** Encoding e imputação de valores ausentes **não são feitos aqui**. Esses tratamentos são responsabilidade exclusiva do `ColumnTransformer` em `Model/train.py`, fitted apenas no conjunto de treino para evitar data leakage.

In [ ]:
df = pd.read_csv('../Dados/clean_data.csv')

print("Dataset carregado com sucesso.")
print(f"  Linhas  : {df.shape[0]:,}")
print(f"  Colunas : {df.shape[1]}")
print()
print("Esperado: 307.511 linhas × 186 colunas (185 features brutas + TARGET)")
print(f"Status  : {'OK' if df.shape == (307511, 186) else 'DIVERGENTE — verificar data_sanitization.py'}")
print()
df.head(3)

## 2. Dimensionalidade e Estrutura

O `clean_data.csv` v3 reúne features de três origens:

| Origem | Quantidade | Descrição |
|---|---|---|
| `application_train.csv` (originais) | ~120 | Dados demográficos, financeiros e de documentação |
| Features derivadas (`create_application_features`) | 8 | Razões financeiras e variáveis temporais |
| Aggregations de `bureau.csv` | 57 | Histórico de crédito externo por cliente |
| **Total features brutas** | **185** | Sem encoding — entrada para o ColumnTransformer |

Após o `ColumnTransformer` (SimpleImputer + OneHotEncoder para categóricas), o modelo opera com **309 features encoded**.

In [ ]:
print("=" * 55)
print("DIMENSOES DO DATASET (clean_data.csv v3)")
print("=" * 55)
print(f"  Registros       : {df.shape[0]:,}")
print(f"  Colunas totais  : {df.shape[1]}")
print(f"  Features brutas : {df.shape[1] - 1}  (excluindo TARGET)")
print()
print(f"Primeiras 5 colunas : {list(df.columns[:5])}")
print(f"Ultimas  5 colunas  : {list(df.columns[-5:])}")
print()
print("Lista completa de colunas:")
for i, col in enumerate(df.columns):
    print(f"  [{i:>3}] {col}")

## 3. Tipos de Variáveis

A identificação dos tipos é essencial para definir as transformações do `ColumnTransformer`:
- **Numéricas** → `SimpleImputer(strategy='median')` — preserva a escala, robusto a outliers
- **Categóricas** → `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`

O `ColumnTransformer` v3 processa **169 features numéricas** e **16 features categóricas**, produzindo **309 features encoded** para o LightGBM.

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"Variaveis numericas   : {len(num_cols)}")
print(f"Variaveis categoricas : {len(cat_cols)}")
print(f"Total (sem TARGET)    : {len(num_cols) + len(cat_cols) - 1}")
print()
print("Variaveis categoricas e suas cardinalidades:")
cat_card = df[cat_cols].nunique().sort_values(ascending=False).rename('n_categorias').to_frame()
display(cat_card)
print()
print("Top 15 menores cardinalidades (todas as colunas):")
display(df.nunique().sort_values().head(15).rename('n_valores_unicos').to_frame())

## 4. Análise de Valores Ausentes

No `clean_data.csv` v3, os valores ausentes refletem o estado **pré-imputação** — eles chegam intactos ao `ColumnTransformer`. Origens dos principais missings:

| Variável | % Missing (raw) | Origem | Interpretação |
|---|---|---|---|
| `EXT_SOURCE_1` | ~56,4% | application_train | Cliente sem score no bureau externo 1 — ausência é preditiva |
| `OWN_CAR_AGE` | ~66% | application_train | Cliente não possui veículo |
| `EXT_SOURCE_3` | ~19,8% | application_train | Cliente sem score no bureau externo 3 |
| `BUREAU_*` (features) | variável | bureau aggregation | Cliente sem histórico em bureau de crédito |
| `EMPLOYED_YEARS` | ~18% | derivada | `DAYS_EMPLOYED = 365243` → convertido para `NaN` (código especial) |

**Decisão de projeto:** Os nulos chegam ao `ColumnTransformer` como `NaN` e são imputados com **mediana do conjunto de treino** (numéricas) ou **moda** (categóricas). A ausência de `EXT_SOURCE_1/3` é capturada indiretamente, pois a mediana da imputação reflete o padrão da população com score disponível.

In [ ]:
missing_abs = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_abs / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Ausentes (n)': missing_abs,
    'Ausentes (%)': missing_pct
})

missing_com_nulos = missing_df[missing_df['Ausentes (n)'] > 0]
print(f"Colunas com valores ausentes: {len(missing_com_nulos)} de {df.shape[1]}")
print()
print("Top 30 variaveis com mais valores ausentes:")
display(missing_com_nulos.head(30))

In [ ]:
top_missing = missing_pct[missing_pct > 0].sort_values(ascending=False).head(25)

if len(top_missing) == 0:
    print("Nenhum valor ausente encontrado no clean_data.csv.")
    print("Verifique se o arquivo foi gerado pelo data_sanitization.py v3.")
else:
    fig, ax = plt.subplots(figsize=(12, max(6, len(top_missing) * 0.35)))
    colors = ['tomato' if v > 50 else ('orange' if v > 30 else 'steelblue') for v in top_missing.values]
    ax.barh(top_missing.index, top_missing.values, color=colors, edgecolor='white')
    ax.axvline(x=30, color='orange', linestyle='--', linewidth=1.2, label='Limite 30%')
    ax.axvline(x=50, color='red', linestyle='--', linewidth=1.2, label='Limite 50%')
    ax.set_xlabel('% de Valores Ausentes')
    ax.set_title('Variaveis com Valores Ausentes (clean_data.csv v3)', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"\nNota: {len(top_missing)} variaveis com missing > 0%.")
    print("Imputacao ocorre em Model/train.py via ColumnTransformer — nunca em data_sanitization.py.")

## 5. Estatísticas Descritivas

Resumo estatístico das variáveis numéricas. Pontos de atenção:
- `DAYS_BIRTH` e `DAYS_EMPLOYED`: valores negativos representam dias **antes** da data de referência
- `DAYS_EMPLOYED = 365243`: código especial (~18% da base) — **não é outlier**, é uma categoria de cliente inativo/aposentado
- `AMT_INCOME_TOTAL` e `AMT_CREDIT`: distribuições com cauda longa à direita (assimetria positiva)
- `EXT_SOURCE_1/2/3`: scores normalizados [0, 1] — valores altos indicam menor risco

In [ ]:
print("Estatisticas descritivas — todas as variaveis numericas:")
display(df.describe().T)

In [ ]:
key_cols = [
    'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_CHILDREN',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]
key_cols_exist = [c for c in key_cols if c in df.columns]

print("Estatisticas descritivas — colunas-chave:")
display(df[key_cols_exist].describe().T)

# Destaque: DAYS_EMPLOYED = 365243
if 'DAYS_EMPLOYED' in df.columns:
    n_especial = (df['DAYS_EMPLOYED'] == 365243).sum()
    pct_especial = n_especial / len(df) * 100
    inadim_especial = df[df['DAYS_EMPLOYED'] == 365243]['TARGET'].mean() * 100
    inadim_demais   = df[df['DAYS_EMPLOYED'] != 365243]['TARGET'].mean() * 100
    print(f"\nDAYS_EMPLOYED = 365243 (codigo especial):")
    print(f"  Ocorrencias         : {n_especial:,} ({pct_especial:.1f}% da base)")
    print(f"  Taxa inadimplencia  : {inadim_especial:.2f}%  (vs {inadim_demais:.2f}% dos demais)")
    print(f"  Decisao: preservado como NaN em EMPLOYED_YEARS — diferenca real de comportamento")

## 6. Features Derivadas — Application Train

A função `create_application_features()` em `DataPipeline/data_sanitization.py` cria **8 features derivadas** a partir das colunas originais do `application_train.csv`. Essas features capturam relações entre variáveis que têm poder preditivo maior do que as variáveis brutas isoladas.

A mesma função é replicada em `Model/predict.py` via `_enrich_input()` para garantir que a inferência online produza exatamente as mesmas features que foram usadas no treino.

| Feature Derivada | Fórmula | Interpretação |
|---|---|---|
| `CREDIT_INCOME_RATIO` | `AMT_CREDIT / AMT_INCOME_TOTAL` | Quanto vezes a renda anual o crédito representa |
| `ANNUITY_INCOME_RATIO` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | Comprometimento mensal da renda com parcelas |
| `CREDIT_ANNUITY_RATIO` | `AMT_CREDIT / AMT_ANNUITY` | Número de parcelas implícito (prazo do empréstimo) |
| `CREDIT_GOODS_RATIO` | `AMT_CREDIT / AMT_GOODS_PRICE` | Proporção do bem financiada (1.0 = 100% financiado) |
| `DOWN_PAYMENT_VALUE` | `AMT_GOODS_PRICE - AMT_CREDIT` | Valor de entrada pago pelo cliente |
| `AGE_YEARS` | `-DAYS_BIRTH / 365.25` | Idade do cliente em anos (sempre positivo) |
| `EMPLOYED_YEARS` | `-DAYS_EMPLOYED / 365.25` | Anos no emprego atual (`365243 → NaN`) |
| `EMPLOYED_AGE_RATIO` | `EMPLOYED_YEARS / AGE_YEARS` | Proporção da vida profissional no emprego atual |

In [ ]:
derived_cols = [
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_ANNUITY_RATIO',
    'CREDIT_GOODS_RATIO', 'DOWN_PAYMENT_VALUE',
    'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_AGE_RATIO'
]
derived_exist = [c for c in derived_cols if c in df.columns]

if not derived_exist:
    print("ATENCAO: Nenhuma feature derivada encontrada.")
    print("Verifique se data_sanitization.py v3 foi executado corretamente.")
else:
    print(f"Features derivadas encontradas: {len(derived_exist)} de {len(derived_cols)} esperadas")
    print()
    display(df[derived_exist].describe().T)

In [ ]:
if derived_exist and 'TARGET' in df.columns:
    corr_derived = df[derived_exist + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values()
    print("Correlacao das features derivadas com TARGET (ordenado):")
    print(corr_derived.to_string())
    print()

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['tomato' if v > 0 else 'steelblue' for v in corr_derived.values]
    ax.barh(corr_derived.index, corr_derived.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Correlacao com TARGET — Features Derivadas de Application Train',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Correlacao de Pearson')
    plt.tight_layout()
    plt.show()
    print()
    print("Interpretacao:")
    print("  CREDIT_INCOME_RATIO positivo: maior endividamento relativo -> maior risco")
    print("  AGE_YEARS negativo: clientes mais velhos -> menor risco")
    print("  EMPLOYED_YEARS negativo: mais tempo de emprego -> menor risco")

## 7. Distribuição da Variável TARGET

`TARGET` é a variável dependente do modelo: `1` indica inadimplência, `0` indica adimplência. O desbalanceamento entre as classes impacta diretamente a estratégia de modelagem:

- **Não usar acurácia como métrica principal** — um modelo trivial que classifica tudo como 0 atingiria ~92%
- **Priorizar Recall** — o custo de aprovar um inadimplente (falso negativo) supera o custo de negar um bom pagador
- **Usar `class_weight='balanced'`** no LightGBM para compensar a razão 11,4:1 entre classes

In [ ]:
target_counts = df['TARGET'].value_counts()
target_pct    = df['TARGET'].value_counts(normalize=True).mul(100).round(2)

print("Distribuicao absoluta (TARGET):")
print(target_counts.rename({0: 'Adimplente (0)', 1: 'Inadimplente (1)'}).to_string())
print()
print("Distribuicao relativa (%):")
print(target_pct.rename({0: 'Adimplente (0)', 1: 'Inadimplente (1)'}).to_string())
print()
print(f"Taxa de inadimplencia     : {target_pct.get(1, 0):.2f}%")
razao = target_counts.get(0, 0) / max(target_counts.get(1, 1), 1)
print(f"Razao de desbalanceamento : {razao:.1f}:1  (adimplentes por inadimplente)")
print(f"scale_pos_weight sugerido : {razao:.2f}  (parametro LightGBM/XGBoost)")

In [ ]:
counts = df['TARGET'].value_counts()
pcts   = df['TARGET'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
bars = axes[0].bar(
    ['Adimplente (0)', 'Inadimplente (1)'],
    [counts.get(0, 0), counts.get(1, 0)],
    color=['steelblue', 'tomato'], edgecolor='white'
)
axes[0].set_title('Contagem por Classe (TARGET)')
axes[0].set_ylabel('Numero de Clientes')
for bar, v in zip(bars, [counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 500,
                 f'{v:,}', ha='center', fontweight='bold', fontsize=11)

# Gráfico de pizza
axes[1].pie(
    [pcts.get(0, 0), pcts.get(1, 0)],
    labels=['Adimplente (0)', 'Inadimplente (1)'],
    autopct='%1.2f%%', colors=['steelblue', 'tomato'],
    startangle=90, explode=[0, 0.06]
)
axes[1].set_title('Proporcao por Classe (TARGET)')

plt.suptitle('Distribuicao da Variavel TARGET — Razao 11,4:1 (desbalanceamento)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Interpretacao:** O dataset apresenta forte desbalanceamento de classes (91,93% adimplentes × 8,07% inadimplentes), razao 11,4:1. Esse padrao e tipico de problemas de concessao de credito. A estrategia adotada no LightGBM v3 e `class_weight='balanced'`, que pondera as amostras de forma inversamente proporcional a frequencia de cada classe, aumentando o peso dos inadimplentes no treinamento e favorecendo o Recall.

## 8. Análise Univariada das Principais Variáveis

Distribuições individuais das variáveis numéricas mais relevantes para risco de crédito, incluindo as features derivadas. As variáveis de valor monetário (`AMT_*`) são truncadas no percentil 99 para melhor visualização das distribuições sem a influência de outliers extremos.

In [ ]:
univar_config = [
    ('DAYS_BIRTH',           'Idade (anos)',            lambda s: s.abs() / 365.25),
    ('AMT_INCOME_TOTAL',     'Renda Total',             None),
    ('AMT_CREDIT',           'Valor do Credito',        None),
    ('AGE_YEARS',            'AGE_YEARS (derivada)',    None),
    ('CREDIT_INCOME_RATIO',  'CREDIT_INCOME_RATIO',     None),
    ('EXT_SOURCE_2',         'EXT_SOURCE_2',            None),
    ('EXT_SOURCE_3',         'EXT_SOURCE_3',            None),
]

# Filtrar apenas colunas existentes
univar_exist = [(col, label, fn) for col, label, fn in univar_config if col in df.columns]

ncols = 3
nrows = (len(univar_exist) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for idx, (col, label, transform) in enumerate(univar_exist):
    data = df[col].dropna()
    if transform is not None:
        data = transform(data)
    # Truncar outliers no p99 para variáveis de valor monetário
    if 'AMT' in col or 'RATIO' in col:
        p99 = data.quantile(0.99)
        data = data[data <= p99]
    axes[idx].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'{label}', fontweight='bold')
    axes[idx].set_xlabel(label)
    axes[idx].set_ylabel('Frequencia')
    # Linha da mediana
    med = data.median()
    axes[idx].axvline(med, color='tomato', linestyle='--', linewidth=1.5, label=f'Mediana: {med:.2f}')
    axes[idx].legend(fontsize=9)

# Ocultar eixos não utilizados
for j in range(len(univar_exist), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Analise Univariada — Principais Variaveis Numericas (linha vermelha = mediana)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Interpretacao:**
> - **Idade (DAYS_BIRTH):** Distribuicao aproximadamente normal entre 20 e 70 anos, concentracao entre 30-50. Clientes mais jovens tendem a maior inadimplencia.
> - **Renda e Credito:** Distribuicoes assimetricas a direita — maioria dos clientes em faixas baixas/medias, com cauda longa para valores altos. A mediana e mais representativa que a media.
> - **AGE_YEARS:** Feature derivada de DAYS_BIRTH — mesma distribuicao, escala mais intuitiva. Usada diretamente em `_enrich_input()` para calculo de EMPLOYED_AGE_RATIO.
> - **CREDIT_INCOME_RATIO:** A maioria dos clientes tem credito entre 2x e 5x a renda anual. Valores muito altos indicam potencial de superendividamento.
> - **EXT_SOURCE_2/3:** Scores externos normalizados [0, 1]. EXT_SOURCE_2 e mais uniforme; EXT_SOURCE_3 e mais concentrada. Ambos correlacionam negativamente com inadimplencia.

## 9. Matriz de Correlação

Análise de correlação linear entre as principais variáveis numéricas e `TARGET`. Inclui tanto features originais quanto features derivadas para mapear as relações mais relevantes para o modelo.

**Nota:** Correlação de Pearson captura apenas relações lineares. O LightGBM captura relações não-lineares, o que explica por que features com correlação linear baixa podem ainda assim ter alta importância no modelo.

In [ ]:
corr_candidates = [
    'TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'CNT_CHILDREN',
    'DAYS_EMPLOYED', 'AGE_YEARS', 'CREDIT_INCOME_RATIO',
    'EMPLOYED_YEARS', 'EMPLOYED_AGE_RATIO', 'AMT_ANNUITY'
]
corr_cols = [c for c in corr_candidates if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Matriz de Correlacao — Features Originais + Derivadas vs TARGET',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelacao com TARGET (ordenado — mais negativo = protetor de risco):")
corr_target = corr_matrix['TARGET'].drop('TARGET').sort_values()
print(corr_target.to_string())

> **Interpretacao:**
> - **EXT_SOURCE_2 e EXT_SOURCE_3** lideram com a maior correlacao negativa com TARGET — scores externos altos protegem contra inadimplencia.
> - **DAYS_BIRTH e AGE_YEARS** correlacionam negativamente com TARGET: clientes mais velhos sao menos propensos a inadimplencia. DAYS_BIRTH (negativo) e AGE_YEARS (positivo) capturam a mesma informacao em escalas distintas.
> - **CREDIT_INCOME_RATIO** correlaciona positivamente: maior endividamento relativo a renda aumenta o risco.
> - **AMT_CREDIT e AMT_ANNUITY** possuem alta correlacao entre si (relacao estrutural de prazo/valor), o que pode gerar multicolinearidade em modelos lineares — nao e problema para o LightGBM.

## 10. Insights — Análise Bivariada

Análise da relação entre variáveis preditivas e `TARGET` para identificar padrões de inadimplência e validar as hipóteses sobre os fatores de risco mais relevantes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. EXT_SOURCE_2 por TARGET
if 'EXT_SOURCE_2' in df.columns:
    for tv, color, label in [(0, 'steelblue', 'Adimplente (0)'), (1, 'tomato', 'Inadimplente (1)')]:
        s = df[df['TARGET'] == tv]['EXT_SOURCE_2'].dropna()
        axes[0, 0].hist(s, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[0, 0].set_title('EXT_SOURCE_2 por TARGET (densidade)', fontweight='bold')
    axes[0, 0].set_xlabel('EXT_SOURCE_2')
    axes[0, 0].set_ylabel('Densidade')
    axes[0, 0].legend()

# 2. Idade (AGE_YEARS ou DAYS_BIRTH) por TARGET
age_col = 'AGE_YEARS' if 'AGE_YEARS' in df.columns else 'DAYS_BIRTH'
transform_age = (lambda s: s) if age_col == 'AGE_YEARS' else (lambda s: s.abs() / 365.25)
if age_col in df.columns:
    for tv, color, label in [(0, 'steelblue', 'Adimplente (0)'), (1, 'tomato', 'Inadimplente (1)')]:
        s = transform_age(df[df['TARGET'] == tv][age_col].dropna())
        axes[0, 1].hist(s, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[0, 1].set_title('Idade (anos) por TARGET (densidade)', fontweight='bold')
    axes[0, 1].set_xlabel('Idade (anos)')
    axes[0, 1].set_ylabel('Densidade')
    axes[0, 1].legend()

# 3. Taxa de inadimplência por faixa de crédito
if 'AMT_CREDIT' in df.columns:
    df_c = df[df['AMT_CREDIT'] <= df['AMT_CREDIT'].quantile(0.99)].copy()
    df_c['faixa_credito'] = pd.cut(
        df_c['AMT_CREDIT'], bins=5,
        labels=['Muito Baixo', 'Baixo', 'Medio', 'Alto', 'Muito Alto']
    )
    taxa_credito = df_c.groupby('faixa_credito', observed=True)['TARGET'].mean() * 100
    axes[1, 0].bar(taxa_credito.index, taxa_credito.values, color='steelblue', edgecolor='white')
    for i, v in enumerate(taxa_credito.values):
        axes[1, 0].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
    axes[1, 0].set_title('Taxa de Inadimplencia por Faixa de Credito', fontweight='bold')
    axes[1, 0].set_ylabel('Taxa de Inadimplencia (%)')
    axes[1, 0].set_xlabel('Faixa de Credito')

# 4. Taxa de inadimplência por número de filhos
if 'CNT_CHILDREN' in df.columns:
    filhos_df = df[df['CNT_CHILDREN'] <= 5].copy()
    filhos_default = filhos_df.groupby('CNT_CHILDREN')['TARGET'].mean() * 100
    axes[1, 1].bar(filhos_default.index.astype(str), filhos_default.values,
                   color='steelblue', edgecolor='white')
    for i, v in enumerate(filhos_default.values):
        axes[1, 1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
    axes[1, 1].set_title('Taxa de Inadimplencia por Numero de Filhos', fontweight='bold')
    axes[1, 1].set_ylabel('Taxa de Inadimplencia (%)')
    axes[1, 1].set_xlabel('Numero de Filhos (ate 5)')

plt.suptitle('Analise Bivariada — Relacao das Variaveis com TARGET', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Insights identificados:**
> 1. **EXT_SOURCE_2 × TARGET:** Inadimplentes concentram-se em scores baixos (0.0–0.4); adimplentes em scores altos. Separacao clara entre distribuicoes — confirma EXT_SOURCE_2 como preditor forte.
> 2. **Idade × TARGET:** Clientes jovens (20–35 anos) tem maior inadimplencia. Acima de 50 anos, o padrao de pagamento e muito mais estavel. Confirmado pelo ranking de gain do LightGBM v3.
> 3. **Credito × TARGET:** Relacao nao-monotonica — creditos de valor medio tendem a maior inadimplencia do que creditos muito altos. Creditos maiores frequentemente exigem mais garantias e analise mais rigorosa.
> 4. **Filhos × TARGET:** Clientes com 1-2 filhos apresentam taxa levemente maior que sem filhos. Acima de 3 filhos a relacao pode inverter (selecao amostral de clientes aprovados com mais dependentes).

## 11. Feature Importance — LightGBM v3

Análise da importância das **309 features encoded** (saída do ColumnTransformer v3) utilizadas pelo modelo **LightGBM v3** em produção (`Model/artifacts/best_model.joblib`), calculada pelo critério **gain**.

**Gain** mede a redução média da impureza (entropia) nos nós onde a feature é usada como critério de split, ponderada pelo número de amostras. É a métrica mais representativa do poder preditivo efetivo de cada variável, diferentemente de:
- `weight` (split count): frequência de uso — favorece features com alta cardinalidade
- `cover`: cobertura de amostras — útil para entender impacto em escala

Os artefatos v3 foram gerados por `docs/HomeCredit_Comparacao_Completa_Modelos_Explicado.ipynb` e `Model/train.py`.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import joblib

# Artefatos v3 — gerados por Model/train.py (LightGBM, ROC-AUC 0.7778, Recall 65.82%)
model        = joblib.load('../Model/artifacts/best_model.joblib')
preprocessor = joblib.load('../Model/artifacts/preprocessor.joblib')

print(f"Modelo carregado : {type(model).__name__}")
print(f"Features encoded : {len(preprocessor.get_feature_names_out())}")
print(f"Features brutas  : {len(joblib.load('../Model/artifacts/features.joblib'))}")

# Importância por gain
gain_values = model.booster_.feature_importance(importance_type='gain')
feat_names  = model.booster_.feature_name()

importance_df = (
    pd.DataFrame({'feature': feat_names, 'gain': gain_values})
    .sort_values('gain', ascending=False)
    .reset_index(drop=True)
)

print("\nTop 10 Features por Gain (LightGBM v3):")
display(importance_df.head(10))

In [ ]:
# Features do formulário Streamlit (nomes com prefixo num__ do ColumnTransformer)
form_set = {
    'num__AMT_INCOME_TOTAL', 'num__AMT_CREDIT', 'num__CNT_CHILDREN',
    'num__DAYS_BIRTH', 'num__DAYS_EMPLOYED',
    'num__EXT_SOURCE_1', 'num__EXT_SOURCE_2', 'num__EXT_SOURCE_3',
    'num__AMT_GOODS_PRICE', 'num__AGE_YEARS', 'num__EMPLOYED_YEARS',
    'num__EMPLOYED_AGE_RATIO', 'num__CREDIT_INCOME_RATIO',
    'num__CREDIT_GOODS_RATIO', 'num__DOWN_PAYMENT_VALUE',
}

top_20 = importance_df.head(20)
colors_bar = ['tomato' if f in form_set else 'steelblue' for f in top_20['feature'][::-1]]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_20['feature'][::-1], top_20['gain'][::-1], color=colors_bar, edgecolor='white')
ax.set_title(
    'Top 20 Features por Importancia (Gain) — LightGBM v3 (309 features encoded)\n'
    '(vermelho = coletadas/derivadas pelo formulario Streamlit | azul = nao coletadas)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Importancia (Gain medio por split)')
plt.tight_layout()
plt.show()

print("\nRanking Top 20 por Gain (LightGBM v3):")
print(f"{'Rank':<6} {'Feature (encoded)':<42} {'Gain':>14}  Status")
print("-" * 78)
for rank, row in enumerate(importance_df.head(20).itertuples(), 1):
    status = 'FORMULARIO' if row.feature in form_set else '-'
    print(f"{rank:<6} {row.feature:<42} {row.gain:>14.2f}  {status}")

> **Interpretacao — Feature Importance LightGBM v3:**
> - **EXT_SOURCE_3 e EXT_SOURCE_2** lideram — scores externos sao os preditores mais poderosos, confirmando a analise de correlacao.
> - **EXT_SOURCE_1** em 3o lugar — apesar de ter 56% de missing no dataset raw, o que e imputado pelo ColumnTransformer ainda carrega sinal preditivo relevante.
> - **DAYS_BIRTH / AGE_YEARS** — o perfil etario e determinante: clientes mais jovens apresentam sistematicamente maior risco.
> - **Features derivadas** (CREDIT_INCOME_RATIO, EMPLOYED_YEARS, EMPLOYED_AGE_RATIO) aparecem no ranking — validando a decisao de feature engineering em data_sanitization.py.
> - **AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE** — o valor e estrutura do financiamento capturam a capacidade de pagamento.

## 12. Cobertura do Formulário Streamlit

O formulário Streamlit (`app/app.py`) coleta **8 campos diretos** do cliente. A função `_enrich_input()` em `Model/predict.py` gera automaticamente **7 features derivadas** a partir desses campos, totalizando **15 features relevantes** mapeadas para o espaço encoded do ColumnTransformer (prefixo `num__`).

Esta seção quantifica o percentual do **gain preditivo total** do LightGBM v3 que é efetivamente capturado pelo formulário.

In [ ]:
# Features capturadas diretamente + derivadas em Model/predict.py -> _enrich_input()
# Nomes com prefixo "num__" pois sao todas numericas apos ColumnTransformer
form_features = {
    'num__AMT_INCOME_TOTAL':    'Renda Total',
    'num__AMT_CREDIT':          'Valor do Credito',
    'num__CNT_CHILDREN':        'Numero de Filhos',
    'num__DAYS_BIRTH':          'Idade (-> DAYS_BIRTH)',
    'num__DAYS_EMPLOYED':       'Tempo de Emprego (-> DAYS_EMPLOYED)',
    'num__EXT_SOURCE_1':        'Score de Bureau 1',
    'num__EXT_SOURCE_2':        'Score de Bureau 2',
    'num__EXT_SOURCE_3':        'Score de Bureau 3',
    'num__AMT_GOODS_PRICE':     'Bem Financiado (= Credito, hardcoded)',
    'num__AGE_YEARS':           'Derivada: -DAYS_BIRTH / 365.25',
    'num__EMPLOYED_YEARS':      'Derivada: -DAYS_EMPLOYED / 365.25',
    'num__EMPLOYED_AGE_RATIO':  'Derivada: EMPLOYED_YEARS / AGE_YEARS',
    'num__CREDIT_INCOME_RATIO': 'Derivada: AMT_CREDIT / AMT_INCOME_TOTAL',
    'num__CREDIT_GOODS_RATIO':  'Derivada: AMT_CREDIT / AMT_GOODS_PRICE (= 1.0)',
    'num__DOWN_PAYMENT_VALUE':  'Derivada: AMT_GOODS_PRICE - AMT_CREDIT (= 0.0)',
}
form_set_local = set(form_features.keys())

rows = []
for f, label in form_features.items():
    mask = importance_df['feature'] == f
    rank = int(importance_df[mask].index[0]) + 1 if mask.any() else None
    gain = float(importance_df[mask]['gain'].values[0]) if mask.any() else 0.0
    rows.append({
        'Label Streamlit': label,
        'Feature (encoded)': f,
        'Rank por Gain': rank,
        'Gain': round(gain, 2)
    })

form_df = (
    pd.DataFrame(rows)
    .sort_values('Rank por Gain', na_position='last')
    .reset_index(drop=True)
)
display(form_df)

total_gain = importance_df['gain'].sum()
form_gain  = importance_df[importance_df['feature'].isin(form_set_local)]['gain'].sum()
n_total    = len(importance_df)
n_form     = importance_df['feature'].isin(form_set_local).sum()

print(f"\nGain total do modelo    : {total_gain:>14,.0f}")
print(f"Cobertura do formulario : {form_gain:>14,.0f}  ({form_gain/total_gain*100:.1f}%)")
print(f"Features capturadas     : {n_form} de {n_total} features encoded do modelo")

> **Interpretacao — Cobertura do Formulario Streamlit:**
> - O formulario captura os **principais preditores por gain** do LightGBM v3 — EXT_SOURCE_1/2/3, DAYS_BIRTH e as features derivadas.
> - **AMT_GOODS_PRICE** esta hardcoded como igual a `AMT_CREDIT` no app (campo removido do formulario). Isso torna `CREDIT_GOODS_RATIO = 1.0` e `DOWN_PAYMENT_VALUE = 0.0` para todos os clientes avaliados via app.
> - **AMT_ANNUITY** nao e coletado — `ANNUITY_INCOME_RATIO` e `CREDIT_ANNUITY_RATIO` chegam como `None` e sao imputados com mediana do treino pelo ColumnTransformer.
> - As features categoricas dummificadas (OCCUPATION_TYPE_*, NAME_EDUCATION_TYPE_* etc.) nao sao coletadas — o modelo usa o vetor zerado, que corresponde a categoria de referencia do OHE.

## 13. Features Ausentes e Estratégia de Inferência

O modelo LightGBM v3 opera com **309 features encoded** geradas pelo ColumnTransformer a partir das 185 features brutas. O formulário Streamlit coleta apenas 8 campos diretos. A função `predict()` em `Model/predict.py` inicializa um DataFrame com todas as **185 features brutas** com valores `None` e preenche apenas os campos fornecidos — o ColumnTransformer então imputa os ausentes.

| Categoria | Exemplos | Estratégia em predict.py | Impacto |
|---|---|---|---|
| Categóricas dummificadas | `OCCUPATION_TYPE_*`, `NAME_EDUCATION_TYPE_*` | Vetor zerado = categoria de referência OHE | Baixo — modelo usa baseline da categoria mais comum |
| Numéricas bureau | `BUREAU_AMT_CREDIT_SUM_SUM` etc. | `None` → imputado com mediana do treino | Baixo — mediana é representativa do cliente sem histórico |
| `AMT_ANNUITY` e razões derivadas | `ANNUITY_INCOME_RATIO`, `CREDIT_ANNUITY_RATIO` | `None` → imputado com mediana do treino | Moderado — campo removido do formulário |
| `AMT_GOODS_PRICE` | `CREDIT_GOODS_RATIO`, `DOWN_PAYMENT_VALUE` | Hardcoded = `AMT_CREDIT` → ratio = 1.0, entrada = 0.0 | Baixo — decisão documentada em CLAUDE.md |
| Flags de presença v2 | `EXT_SOURCE_1_MISSING` (não existe em v3) | N/A — ausência capturada via imputação pela mediana | Nulo — não existe no ColumnTransformer v3 |

**Fluxo de inferência online:**
```
Usuario preenche 8 campos no formulario Streamlit
    ↓  app/app.py  →  predict(dict_inputs)
    ↓  Model/predict.py  →  _enrich_input()  (replica create_application_features)
DataFrame com 185 features brutas (campos nao fornecidos = None)
    ↓  preprocessor.transform()  →  309 features encoded (None -> mediana/moda)
    ↓  best_model.predict_proba()
{"prediction": 0|1, "probability": float}
    ↓  app/app.py
BAIXO RISCO (< 30%) | MEDIO RISCO (30-70%) | ALTO RISCO (>= 70%)
```

In [ ]:
# Verificar quais features do formulario existem no modelo
model_features_set = set(importance_df['feature'].tolist())

print("Verificacao de cobertura das features do formulario no modelo v3:")
print(f"{'Feature (encoded)':<42} {'No modelo?':<12} {'Rank Gain':<12} {'Gain':>10}")
print("-" * 80)
for f, label in form_features.items():
    in_model = f in model_features_set
    if in_model:
        mask = importance_df['feature'] == f
        rank = int(importance_df[mask].index[0]) + 1
        gain = float(importance_df[mask]['gain'].values[0])
        print(f"{f:<42} {'SIM':<12} {rank:<12} {gain:>10.1f}")
    else:
        print(f"{f:<42} {'NAO':<12} {'N/A':<12} {'0.0':>10}")

print()
n_encontradas = sum(1 for f in form_features if f in model_features_set)
print(f"Features do formulario encontradas no modelo: {n_encontradas} de {len(form_features)}")
print()
print("Features nao encontradas (podem estar com nome diferente no ColumnTransformer v3):")
nao_encontradas = [f for f in form_features if f not in model_features_set]
for f in nao_encontradas:
    print(f"  - {f}  ({form_features[f]})")

## 14. Conclusões

---

### Resumo dos Principais Achados

#### Perfil do Dataset (clean_data.csv v3)
- **307.511 registros × 186 colunas** (185 features brutas + TARGET)
- Composicao: ~120 features originais do application_train + 8 features derivadas + 57 aggregations do bureau
- Variavel alvo binaria com **forte desbalanceamento**: 91,93% adimplentes × 8,07% inadimplentes, razao 11,4:1
- Justifica `class_weight='balanced'` no LightGBM v3 para aumentar o peso dos inadimplentes no treinamento

#### Valores Ausentes e Decisoes de Projeto
- **DAYS_EMPLOYED = 365243** preservado como `NaN` em `EMPLOYED_YEARS` — e um codigo especial presente em ~18% da base. Clientes com esse valor tem taxa de inadimplencia diferente (5,4% vs 8,66%), o que confirma que nao deve ser tratado como outlier nem imputado com mediana
- **EXT_SOURCE_1** (~56% missing no raw) e **EXT_SOURCE_3** (~20% missing): ausencia estrutural e preditiva, nao erro de coleta. Nulos chegam ao ColumnTransformer como `NaN` e sao imputados com mediana do treino
- **Features bureau** (`BUREAU_*`): nulos representam ausencia de historico em bureau de credito, imputados com `0` em data_sanitization.py
- **OCCUPATION_TYPE** (~31% nulo no raw): preenchido com `'UNKNOWN'` antes do encoding em data_sanitization.py

#### Preditores Mais Fortes (confirmados pela Feature Importance do LightGBM v3)
1. **EXT_SOURCE_3** — maior gain individual; score de bureau externo com maior poder discriminativo
2. **EXT_SOURCE_2** — segundo maior gain; complementa o EXT_SOURCE_3
3. **EXT_SOURCE_1** — terceiro maior gain; apesar dos 56% de missing, ainda carrega sinal relevante
4. **DAYS_BIRTH / AGE_YEARS** — perfil etario: clientes mais jovens tem maior risco sistematico
5. **Features derivadas** (CREDIT_INCOME_RATIO, EMPLOYED_YEARS, EMPLOYED_AGE_RATIO) — validam o feature engineering de create_application_features()

#### Formulario Streamlit e Cobertura do Modelo
- O formulario coleta 8 campos diretos; `_enrich_input()` gera 7 features derivadas adicionais
- As 15 features coletadas/derivadas cobrem os principais preditores por gain do LightGBM v3
- **Restricoes de design:** `AMT_GOODS_PRICE = AMT_CREDIT` (hardcoded), `AMT_ANNUITY` nao coletado (imputado com mediana)
- Features categoricas (educacao, tipo de emprego, organizacao) nao coletadas — modelo usa vetor zerado (categoria de referencia do OHE)

#### Modelo de Referencia em Producao — LightGBM v3

| Metrica | LightGBM v3 | Observacao |
|---|---|---|
| ROC-AUC | 0,7778 | Melhor entre os 5 modelos treinados |
| Recall | 65,82% | Metrica prioritaria — minimiza inadimplentes aprovados |
| Precision | 18,90% | Baixa precisao e esperada com class_weight balanced |
| F1-Score | 29,37% | Balance entre Recall e Precision |
| Average Precision | 27,68% | Area sob curva Precision-Recall |

#### Proximo Passo
Executar **`Model/evaluation.ipynb`** para avaliacao completa do LightGBM v3: curva ROC, curva Precision-Recall, matriz de confusao, analise de limiar de decisao e comparacao com os 4 outros modelos treinados (Dummy, Logistic Regression, Random Forest, XGBoost).

---
*Analise exploratoria concluida. Artefatos v3 em `Model/artifacts/`. Pipeline de dados documentado em `DataPipeline/data_preparation.ipynb`.*